# Person 2 — Preprocessing, duplicate audit, and leakage-safe split

Part of the six-person Basil Leaf ML pipeline. Run the numbered notebooks in order. This notebook states its inputs, produces a concrete handoff in `parts/artifacts`, and does not overwrite the complete project's `outputs/` results.

## Responsibility
Decode images safely, correct camera orientation, convert to RGB, reject invalid files, remove exact pixel duplicates, form perceptual-similarity groups, and create one untouched grouped test split.

**Input:** Person 1 inventory and `data/raw/`  
**Output:** `02_clean_manifest.csv`, `02_split_manifest.csv`, and `02_preprocessing_report.json`

In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "Basil_Leaf_ML_Workflow.ipynb").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the project folder or parts folder.")
DATA_DIR = ROOT / "data" / "raw"
ARTIFACTS = ROOT / "parts" / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
SEED = 42
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
FOLDERS = {
    "Amravati_Region_Basil_Plant_Healthy": ("Healthy", 31),
    "Nagpur_Region_Basil_Plant_Healthy": ("Healthy", 473),
    "Pune_Region_Basil_Plant_Healthy": ("Healthy", 146),
    "Basil_Plant_Unhealthy": ("Unhealthy", 481),
}
print("Project:", ROOT)
print("Python:", sys.executable)
from PIL import Image, ImageOps
from scipy.fft import dctn
from sklearn.model_selection import StratifiedGroupKFold
import hashlib, warnings
Image.MAX_IMAGE_PIXELS=25_000_000

Project: D:\SLIIT\projectr\Dataset_Train
Python: D:\SLIIT\projectr\Dataset_Train\.venv\Scripts\python.exe


In [2]:
if not (ARTIFACTS/'01_inventory.csv').exists():
    raise FileNotFoundError("Run Person 1 notebook first.")
inventory=pd.read_csv(ARTIFACTS/'01_inventory.csv')
rows=[]; rejected=[]
for folder,(label,_) in FOLDERS.items():
    for path in sorted((DATA_DIR/folder).rglob("*")):
        if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS: continue
        try:
            with warnings.catch_warnings():
                warnings.simplefilter('error',Image.DecompressionBombWarning)
                with Image.open(path) as opened:
                    if min(opened.size)<16: raise ValueError('image smaller than 16 pixels')
                    opened.load(); image=ImageOps.exif_transpose(opened).convert('RGB')
            pixel_hash=hashlib.sha256(str(image.size).encode()+image.tobytes()).hexdigest()
            gray=np.asarray(image.resize((32,32)).convert('L'),dtype=float)
            low=dctn(gray,norm='ortho')[:8,:8].ravel()[1:]
            bits=low>np.median(low)
            phash=sum(int(bit)<<i for i,bit in enumerate(bits))
            rows.append({"path":path.relative_to(DATA_DIR).as_posix(),"label":label,"width":image.width,"height":image.height,"pixel_sha256":pixel_hash,"phash":phash})
        except Exception as exc: rejected.append({"path":str(path),"reason":str(exc)})
frame=pd.DataFrame(rows)
conflicts=frame.groupby('pixel_sha256').label.nunique()
if (conflicts>1).any(): raise ValueError('Identical pixels have conflicting labels.')
parent=list(range(len(frame)))
def find(a):
    while parent[a]!=a:
        parent[a]=parent[parent[a]]; a=parent[a]
    return a
def union(a,b): parent[find(b)]=find(a)
hashes=frame.phash.tolist(); exact={}
for i,row in frame.iterrows():
    if row.pixel_sha256 in exact: union(exact[row.pixel_sha256],i)
    else: exact[row.pixel_sha256]=i
    for j in range(i):
        if (int(hashes[i])^int(hashes[j])).bit_count()<=4: union(i,j)
frame['split_group']=[f'group-{find(i):05d}' for i in range(len(frame))]
duplicates=int(frame.pixel_sha256.duplicated().sum())
frame=frame.drop_duplicates('pixel_sha256').reset_index(drop=True)
y=frame.label.to_numpy(); groups=frame.split_group.to_numpy()
dev,test=next(StratifiedGroupKFold(5,shuffle=True,random_state=SEED).split(frame,y,groups))
frame['split']='development'; frame.loc[test,'split']='test'
if set(frame.loc[dev,'split_group']) & set(frame.loc[test,'split_group']): raise AssertionError('Group leakage')
if set(frame.loc[dev,'pixel_sha256']) & set(frame.loc[test,'pixel_sha256']): raise AssertionError('Duplicate leakage')
frame.to_csv(ARTIFACTS/'02_clean_manifest.csv',index=False)
frame[['path','label','split_group','split']].to_csv(ARTIFACTS/'02_split_manifest.csv',index=False)
report={"decoded":len(rows),"unique":len(frame),"duplicates_removed":duplicates,"rejected":rejected,"groups":int(frame.split_group.nunique()),"development":int((frame.split=='development').sum()),"test":int((frame.split=='test').sum()),"class_counts":frame.label.value_counts().to_dict()}
(ARTIFACTS/'02_preprocessing_report.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
display(frame.groupby(['split','label']).size().rename('images').reset_index()); print(report)

,split,label,images
0,development,Healthy,366
1,development,Unhealthy,351
2,test,Healthy,92
3,test,Unhealthy,88


{'decoded': 898, 'unique': 897, 'duplicates_removed': 1, 'rejected': [], 'groups': 759, 'development': 717, 'test': 180, 'class_counts': {'Healthy': 458, 'Unhealthy': 439}}


## Handoff to Person 3
Give Person 3 the clean manifest and fixed split membership. The test rows must remain untouched during feature and model design.